In [30]:
import pandas as pd
# načítanie
data1 = pd.read_csv("data/stocks/A.csv")
data2 = pd.read_csv("data/stocks/AA.csv")


data1["Date"] = pd.to_datetime(data1["Date"])
data2["Date"] = pd.to_datetime(data2["Date"])

# nastavenie indexu na Date
data1 = data1.set_index("Date")
data2 = data2.set_index("Date")

# konverzia Date na datetime
data1.columns = pd.MultiIndex.from_product([["A"], data1.columns])
data2.columns = pd.MultiIndex.from_product([["AA"], data2.columns])

# join podľa indexu (Date)
joined_data = data1.join(data2, how="inner", lsuffix="_A", rsuffix="_AA")

#joined_data = joined_data.sort_index()  # zoradenie podľa dátumu
#joined_data = joined_data.reset_index()  # pridá stĺpec "Date" a nový index 0..n-1


In [31]:
#joined_data.head()

#joined_data['A'].iloc[10]['Open']
#joined_data.iloc[5:10]

#print(joined_data.head(1))
#print(joined_data.columns)
#len(joined_data['A'])
#len(joined_data['AA'])
#print(joined_data['AA'].iloc[5124])
#joined_data['A']
#joined_data['A'].iloc[0]

In [32]:
# #environment
# from gymnasium.utils.env_checker import check_env
# from trading import Trading

# #joined_data.head()
# # This will catch many common issues
# environment = Trading(data=joined_data, init_cash=1000)
# try:
#     check_env(environment)
#     print("Environment passes all checks!")
# except Exception as e:
#     print(f"Environment has issues: {e}")

In [33]:
import logging

logging.basicConfig(
    filename='my_logfile.log',   # názov log súboru
    level=logging.INFO,          # úroveň logovania (DEBUG, INFO, WARNING, ERROR)
    format='%(asctime)s - %(levelname)s - %(message)s',
    filemode='w'                 # 'w' prepíše súbor, 'a' dopíše
)

# episodes = 2
# for episode in range(1,episodes+1):
#     state = environment.reset()
#     done = False
#     score = 0
#     day=0
    
#     while not done:
        
#         action=environment.action_space.sample()
#         n_state,reward,done,_,info = environment.step(action)
#         score+=reward
#         logging.info(f"{n_state},{score}, {day}")
#         day+=1
        
#     print(f'Episode {episode}: Score:{score}')

In [ ]:
# DQN tutorial
# https://github.com/nicknochnack/OpenAI-Reinforcement-Learning-with-Custom-Environment/blob/main/OpenAI%20Custom%20Environment%20Reinforcement%20Learning.ipynb

import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from trading import Trading
from gymnasium.spaces import flatten

# obs = {
#             "price_history": 10,
#             "holdings": {
#                 "cash": 10,
#                 "stock1": {
#                     "holding_stock":10,
#                     "price_for_one_stock":10 
#                     },
#                 "stock2": {
#                     "holding_stock":10,
#                     "price_for_one_stock": 10 
#                     }},
#             "portfolio_price": 10
        
#         }

            
# # states = flatten(environment.observation_space,obs).shape
# # actions = np.array([0,0]).shape
# # print(actions, states)

# states = flatten(environment.observation_space,obs).shape
environment = Trading(data=joined_data, init_cash=1000)
states = (7,)
actions = 2



In [ ]:
#https://keras.io/examples/rl/deep_q_network_breakout/
import os

os.environ["KERAS_BACKEND"] = "tensorflow"

import keras
from keras import layers

import gymnasium as gym
from gymnasium.wrappers import AtariPreprocessing,FrameStackObservation
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import clone_model
import random
num_actions = 4

# Configuration parameters for the whole setup
seed = 42
gamma = 0.99  # Discount factor for past rewards
epsilon = 1.0  # Epsilon greedy parameter
epsilon_min = 0.1  # Minimum epsilon greedy parameter
epsilon_max = 1.0  # Maximum epsilon greedy parameter
epsilon_interval = (
    epsilon_max - epsilon_min
)  # Rate at which to reduce chance of random action being taken
batch_size = 32  # Size of batch taken from replay buffer
max_steps_per_episode = 10000
max_episodes = 10  # Limit training episodes, will run until solved if smaller than 1




def create_q_model(input_dim, num_actions):
    return keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.Dense(128, activation="relu"),
        layers.Dense(num_actions, activation="linear")
    ])

def policy(best_action, env):
    
    if random.random()>0.5:
        return best_action
        
    
    return env.action_space.sample()

# The first model makes the predictions for Q-values which are used to
# make a action.
actor_model = create_q_model(8,12)


actor_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='mse'   # pre Q-learning typicky mean squared error
)
critic_model = clone_model(actor_model)
critic_model.set_weights(actor_model.get_weights())

# Build a target model for the prediction of future rewards.
# The weights of a target model get updated every 10000 steps thus when the
# loss between the Q-values is calculated the target Q-value is stable.
#model_target = create_q_model(7,actions)

In [ ]:

episodes = 2
for episode in range(1,episodes+1):
    #
    #                   /--> a21 |                         tQ(s2,a21) |                        Q(s2,a21) 
    # |s1| -- a11 --> s2 --> a22 | critic -- generate -->  tQ(s2,a22) | actor -- generate -->  Q(s2,a22)    
    #                   \--> a23 |                         tQ(s2,a23) |                        Q(s2,a23) 
    #
    # Q(s1,a11) = reward + gamma * max(actor.generate())
    # target_Q(s1,a11) = reward + gamma * max(critic.generate())
    #
    # actor_learn = loss(target_Q(s1,a11) - Q(s1,a11))
    
    environment.reset()
    done = False
    score = 0
    day=0
    
    # first day state_prew doesnt exist yet
    action=environment.action_space.sample()
    state_current,reward,_,_,_ = environment.step(action)
    Q_prev=0
    target_update_freq = 10
    step = 0
    # we have next state and reward given by taking an action
    while not done:      
        step+=1
        #training
        
        state = flatten(environment.observation_space,state_current)
        state = state[np.newaxis, :]
        
        Q_values = actor_model.predict(state,verbose=0)[0]
        Q_targets = critic_model.predict(state,verbose=0)[0]
        
        stock_Q_values = [Q_values[0:6],Q_values[6:12]]
        
        best_Q_values = [ reward + gamma * max(stock_action_values) for stock_action_values in stock_Q_values]
        
        
        best_action = [np.argmax(stock_Q_values[0]),np.argmax(stock_Q_values[1])]
        action = policy(best_action, environment)
        # perform action and thus get new state
        state_current,reward,terminated,_,_ = environment.step(action)

        
        logging.info(f"{state_current},{score}, {day}")
        day+=1
        
        if step % target_update_freq == 0:
          critic_model.set_weights(actor_model.get_weights())
          logging.info(f"{state_current},{score}, {day}")
        
        loss = actor_model.train_on_batch(state, Q_targets)
        score +=loss
        
        if terminated:
          done=True
    
    actor_model.save("actor_model.h5")
    print(f'Episode {episode}: Score:{score}')



Episode 1: Score:0.0


KeyboardInterrupt: 

In [ ]:
from trainer import Trainer

#tr = Trainer(config="./example_config.json",data_dir="./data/stocks")
#tr.train(stock_names=["A","AA"], descriptors = ["Open"])

flatten(environment.observation_space,state_current)